# 08 — QGIS Exercise: LiDAR DTM, Hillshading, and α–β Run-out for Debris Flows

**Short course:** *Geomorphological Hazards of Slopes* &nbsp;•&nbsp; University of Silesia in Sosnoviec &nbsp;•&nbsp; 2 ECTS

*Lecturer: Ola Fredin*

---

Operational landslide mapping is done in GIS such as **QGIS**. Field geologists, consultants, and national geological surveys all work in a desktop GIS, because that is where the geospatial data lives, where the styling is fast, and where the symbology is shareable across a team. This notebook is therefore a **guided QGIS lab**, not a Python walkthrough. You will open a real 1 m LiDAR DTM, render a hillshade, digitise a small inventory of debris-flow tracks and deposits, and finish by predicting their run-out lengths using an α–β relationship.

The notebook itself acts as your **lab handout**: every step in QGIS gets its own markdown block with menu paths, parameter values, and a slot for a screenshot. The only Python in the whole notebook is a short cell at the end that converts the β angles you measured in QGIS into predicted run-out distances.

> ⚠️ **Unpublished material — do not circulate.** The α–β coefficients used in §6 and §7 of this notebook come from Nora Andreassen's *unpublished* MSc thesis (NTNU, June 2026). They are shared here with the author's permission for course use only. **Do not redistribute, repost, or cite outside the course** until the work is formally published. If you want to refer to the result in your own work, contact Ola Fredin first.

## About this notebook

**Learning objectives.** By the end of this exercise the student will be able to:

1. Load a 1 m bare-earth LiDAR DTM into QGIS and render a multi-azimuth hillshade for landslide mapping.
2. Digitise debris-flow source areas, tracks, and deposits as point/line/polygon shapefiles with a consistent attribute schema.
3. Reclassify a slope raster into 2° bins and isolate the 19–21° class to locate the **β-point** — the first position along the flow path where the terrain flattens to 20°.
4. Extract β, *H*, and observed runout *L* from the DTM and the mapped features.
5. Apply the α–β method to predict run-out length, and compare predicted vs. observed values.

**Prerequisites.** Notebook 07 (debris-flow run-out theory) for the mechanical context. No prior QGIS experience is assumed; every step is spelled out.

**Software.** QGIS 3.34 LTR or newer (free, [qgis.org/download](https://qgis.org/download)). No paid plugins required. The *Profile Tool* plugin is useful but optional.

## 1. Data — what you need on disk before you start

You will work with a 1 m bare-earth LiDAR DTM and a 0.5 m CIR orthophoto for the study area, plus an (initially empty) project folder for the shapefiles you will create.

**Study area.** Skamsdalen, central Norway — a glacially scoured U-shaped valley with approximately **1000 m of local relief**. The northern valley side displays numerous active debris flows originating in till that is about **0.5–4 m thick**. This till was deposited during the Weichselian deglaciation approximately **10 000 years ago** and is now being mobilised by gravitational processes.

**DTM tile.**

- Source: [hoydedata.no](https://hoydedata.no) (Kartverket national LiDAR archive — free download for Norway)
- Filename: e.g. `skamsdalen_dtm_1mC.tif`
- Resolution: 1 m (bare-earth, last-return classification)
- CRS: **EPSG:5973** (ETRS89 / ETRS89 / UTM zone 33N + NN2000 height + NN2000 height)
- Coverage: Skamsdalen valley and the debris-flow-bearing northern slope

**CIR orthophoto.**

- Source: [norgeibilder.no](https://norgeibilder.no) (Geovekst programme — free download for Norway)
- Filename: e.g. `skamsdalen_cir_0.5mC.tif`
- Resolution: 0.5 m
- Bands: Near-infrared (band 1), Red (band 2), Green (band 3) — a **false-colour infrared (CIR)** composite
- Importantly: active debris-flow tracks lack dense vegetation, so they appear in **cyan** against the **red** of undisturbed vegetated terrain. This makes it much easier to identify debris-flow starting points and track margins before you digitise on the hillshade.
- CRS: **EPSG:5973** (same as DTM — no reprojection needed)

**Suggested project folder structure.**

```
qgis_lab_08/
  data/
    skamsdalen_dtm_1mC.tif      ← the 1 m LiDAR DTM from hoydedata.no
    skamsdalen_cir_0.5mC.tif    ← the 0.5 m CIR orthophoto from norgeibilder.no
  layers/                       ← created during the exercise
    debris_flow_sources.shp
    debris_flow_tracks.shp
    debris_flow_deposits.shp
  Skamsdalen_project.qgz                   ← the QGIS project file
```

## 2. Step 1 — Load the LiDAR DTM and CIR orthophoto into QGIS

**Goal.** Get the 1 m DTM and 0.5 m CIR orthophoto into a fresh QGIS project with CRS set to **EPSG:5973 (ETRS89 / UTM zone 33N + NN2000 height)**, and apply a colour ramp so you can check the elevation data before proceeding.

**Menu path.**

1. *Project → New*, then save immediately as `Skamsdalen_project.qgz` inside your `qgis_lab_08/` folder.
2. *Layer → Add Layer → Add Raster Layer…* → navigate to `data/skamsdalen_dtm_1mC.tif` → Add.
3. Confirm the project CRS in the bottom-right corner of the QGIS window: it should read **EPSG:5973**. If it does not, right-click the DTM layer → *Set CRS → Set Project CRS from Layer*.
4. Right-click the DTM layer → *Properties → Symbology* → set render type to **Singleband pseudocolor**, colour ramp **Turbo** or **Viridis**, classify with 7–10 classes. Click OK.
5. Now add the CIR orthophoto: *Layer → Add Layer → Add Raster Layer…* → navigate to `data/skamsdalen_cir_0.5mC.tif` → Add.
6. Right-click the CIR layer → *Properties → Symbology* → confirm the band assignment: Band 1 → Red channel, Band 2 → Green channel, Band 3 → Blue channel. This renders the near-infrared band as red, making vegetated slopes appear bright red and debris-flow scars cyan. Click OK.
7. In the Layers panel, drag the CIR layer **above** the DTM so it is visible by default. You can toggle it on/off as needed when mapping.


## 3. Step 2 — Create a hillshade

**Goal.** Produce a greyscale hillshade that brings the morphology of debris-flow tracks and lobes into relief. We keep one hillshade as the working basemap and optionally stack a second with a different sun azimuth to catch scarps running parallel to the first illumination direction.

**Menu path.**

1. *Raster → Analysis → Hillshade…*
2. Input layer = your DTM (`skamsdalen_dtm_1m.tif`); Z factor = **1.0** (CRS is metric — EPSG:5973 — so no vertical exaggeration needed).
3. Azimuth = **315°**, vertical angle = **45°** — the classical NW illumination that works well for Norwegian valley topography.
4. Output = `layers/hillshade_315.tif`. Click Run.
5. *(Optional but recommended)* Repeat with azimuth **45°** to catch linear features (scarps, lateral levées) running NW–SE that are invisible under NW illumination. Save as `layers/hillshade_045.tif`. In the Layers panel, stack it on top of the first hillshade at **50 % transparency**.
6. Move the hillshade(s) **below** your pseudocolour DTM; set the DTM blending mode to **Multiply** at **60 % opacity** to achieve the classic colour-hillshade look. Move the CIR layer to the top so you can toggle it on and off over the hillshade without disturbing the blend.

**Parameters / things to write down.**

- Sun azimuth(s) used: 315° (primary); 45° (optional second)
- Sun altitude: 45°
- Z-factor: 1.0 (required for metric CRS — do not change)

![alt text](Blending-hillshade-dem.png)

Here we are blending the pseudocolor DTM with the Hillshade. You can also do the same with the CIR orthophoto.

![alt text](Blending-hillshade-CIR.png)

Here we are blending the CIR image with the Hillshade.

## 4. Step 3 — Map debris-flow features and record measurements as you go

**Goal.** Build a small inventory of 3–5 debris flows visible on the hillshade. Each flow gets three pieces of geometry: a **source point** at the head, a **track line** running down the channel, and a **deposit polygon** at the runout fan. Crucially, you will read the few numbers the α–β method needs **on the fly** and type them straight into a small **Excel sheet** — there is no separate elevation-extraction step later.

> Keep the spreadsheet `qgis_lab_08/debris_flow_measurements.xlsx` open next to QGIS. Every time you click a feature, read its elevation with the **Identify Features** tool on the DTM and type it into the sheet immediately. We deliberately **skip** the more automated approach of overlaying the digitised points on the DEM raster and batch-sampling elevations (*Sample raster values*). For a handful of flows in a teaching exercise, typing four numbers per flow is simpler, faster to understand, and leaves nothing hidden. For a real inventory of hundreds of flows you would of course automate the sampling instead.

### 4.1 Create the three shapefiles

The shapefiles now only need to carry **geometry** and a **`flow_id`** join key — the elevations live in the Excel sheet, not in the attribute tables.

**Menu path.** Repeat the following three times, once per layer:

1. *Layer → Create Layer → New Shapefile Layer…*
2. Set filename, geometry type, **CRS = EPSG:5973 (ETRS89 / UTM zone 33N + NN2000 height)**, and a minimal attribute schema.

| Layer | Geometry | Fields |
|---|---|---|
| `debris_flow_sources.shp`  | Point   | `flow_id` (int), `notes` (text 100) |
| `debris_flow_tracks.shp`   | LineString | `flow_id` (int), `notes` (text 100) |
| `debris_flow_deposits.shp` | Polygon | `flow_id` (int), `notes` (text 100) |

`flow_id` is the join key — each flow gets a unique integer, and the same integer is used across all three layers **and** in the Excel sheet.

### 4.2 Digitise each flow and fill the spreadsheet

Work **one flow at a time**, and for each flow fill one row of `debris_flow_measurements.xlsx`:

1. *Select a layer in the Layers panel → toggle editing (pencil icon) → use the Add Feature tool.*
2. **Source point** — place it at the head of the flow. With the **Identify Features** tool (i-icon) click the same spot on the DTM, read the elevation, and type it into the **`z_source`** column.
3. **Track line** — digitise down the channel. (You will find the β-point on this line in Step 4 and fill **`z_beta`** then.)
4. **Deposit polygon** — draw it over the fan. Identify the DTM at the toe (lowest extent) and type the elevation into **`z_toe`**.
5. Keep the **`flow_id`** identical across the three layers and the spreadsheet row.
6. Save edits frequently (Ctrl + S) and save the spreadsheet too.

The `L_beta` and `L_obs` distance columns are filled in Step 5 with the Measure Line tool. The grey `H` and `beta_deg` columns are computed automatically by the sheet as a sanity check — you never type into them.

**Tips.**

- Use the **Snapping toolbar** so the track line snaps to the source point and the deposit polygon — keeps the geometry tidy.
- Aim for 3–5 mapped flows total, spanning small (a few hundred metres long) to large ( > 1 km) so the α–β analysis has some spread.

![alt text](Mapping_runout_QGIS.png)
Digitizing a source (starting point) a debris track, and debris flow deposit.

## 5. Step 4 — Create a slope map and identify the β-point

**Goal.** Produce a slope raster from the DTM, classify it into 2° bins, then isolate the 19–21° class so that each debris-flow track shows a narrow highlighted band exactly where the local terrain first flattens to 20°. That crossing point is the **β-point** for the flow — the boundary between the transport and deceleration phases (Norem & Sandersen 2012).

### 5.1 Compute the slope raster

1. *Raster → Analysis → Slope…*
2. Input layer = `skamsdalen_dtm_1mC.tif`; **Z factor = 1.0** (metric CRS — do not change); output in **degrees**.
3. Save as `layers/slope_deg.tif`. Click Run.
4. Move the slope layer **above** the hillshade in the Layers panel; give it a quick single-band pseudocolour check (Symbology → Viridis) to confirm slope values look sensible. Expect 40–85° on the steep upper headwalls and < 5° on the valley floor.

![alt text](Slope_Viridis.png)

### 5.2 Reclassify into three classes

1. *Processing Toolbox → search "Reclassify by table"* (under QGIS → Raster analysis).
2. Input raster: `slope_deg.tif`.
3. In the **Reclassification table**, add just three rows. The middle class brackets the 20° run-out threshold:

   | Minimum | Maximum | Value | Meaning |
   |---------|---------|-------|---------|
   | 0       | 19      | 1     | gentle — below the threshold band |
   | **19**  | **21**  | **2** | the β-band — terrain flattening through ~20° |
   | 21      | 90      | 3     | steep — above the threshold band |

4. Range boundaries: set to **min ≤ value < max** (default).
5. Output: `layers/slope_reclass.tif`. Click Run.

### 5.3 Symbolize — show only the 19–21° band

1. Right-click `slope_reclass.tif` → *Properties → Symbology*.
2. Render type: **Paletted / Unique values** → Click **Classify**.
3. QGIS will list values 1–3. Right-click classes **1 (<19°)** and **3 (>21°)** → *Change opacity → 0 %* (fully transparent). Alternatively, delete those two rows and keep only value 2.
4. Set the colour for value 2 (19–21°) to a high-contrast colour such as **red**.
5. Click OK. Only pixels in the 19–21° slope range should now be visible as a narrow coloured band on the hillshade.

> **Why 19–21° instead of exactly 20°?** At 1 m resolution a single-pixel-wide 20° contour would be almost invisible. Displaying a 2°-wide band centred on the threshold (19–21°) gives a band wide enough to see while still pinpointing the transition. The β-point sits where this band first crosses your mapped track as you follow it downslope.

### 5.4 Locate the β-point for each flow

1. Toggle the reclassified slope layer on over the hillshade.
2. Zoom into each mapped debris-flow track (use the **track lines** from Step 3 as a guide).
3. Follow the track **downslope** from the source. The first point where the **red band (19–21°)** crosses the track is the **β-point**.
4. Use the **Identify Features** tool on the DTM to read the elevation at that point and type it straight into the **`z_beta`** column of `debris_flow_measurements.xlsx` for that flow.
5. Optional: add a temporary point marker at each β-point (*Layer → Create Temporary Scratch Layer*, geometry = Point) so you can snap to it precisely when measuring distances.

![alt text](beta_reclass.png)

Here we see that the debris flow track crosses many spots where the slope breaks around $20 \circ$. However, quite low down in the slope, it breaks consistently below $20 \circ$, this is the area we want to use for our **`z_beta`** readout.

**Parameters / things to write down.**

- Confirm Z factor = 1.0 (you will notice wrong results immediately if this is wrong).
- Note whether the β-point falls on the main channel, a tributary, or on the open slope — this affects how you measure the β-angle in the next step.


## 6. Step 5 — Complete the measurements in the spreadsheet

By now your Excel sheet already holds `z_source`, `z_beta`, and `z_toe` for each flow — you typed them in as you digitised (Steps 3–4). All that is left is the two **horizontal distances**, which come from the Measure Line tool, not from the DEM.

The α–β method needs three derived numbers per flow; the spreadsheet builds them from what you record:

- **β-angle** — the straight-line inclination from the source to the β-point:
  $\beta = \arctan\!\left(\dfrac{z_{\mathrm{source}} - z_{\beta}}{L_{\beta}}\right)$,
  where $L_{\beta}$ is the **horizontal** distance from source to β-point. *(Computed for you in the `beta_deg` column.)*
- **Fall height** $H = z_{\mathrm{source}} - z_{\mathrm{toe}}$ — the full vertical drop from source to deposit toe. *(Computed for you in the `H` column.)*
- **Observed runout** $L_{\mathrm{obs}}$ — horizontal distance from source to deposit toe. Used in §7 to check the model.

**Measure the two distances.**

1. Use the **Measure Line** tool (*View → Measure → Measure Line*, or Ctrl+Shift+M) with snapping enabled.
2. Click the source point → click the β-point → read the distance from the status bar and type it into the **`L_beta`** column.
3. Click the source point → click the deposit toe → type the distance into the **`L_obs`** column.
4. The CRS is EPSG:5973 (metric projected), so distances are already in metres — no conversion needed.

**Check.** The β-point must lie *between* the source and the deposit toe — if $L_{\beta} > L_{\mathrm{obs}}$ you have either placed the β-point wrongly or measured in the wrong direction.

> **Why no raster sampling?** Earlier versions of this lab overlaid the digitised points on the DTM and batch-extracted elevations with *Sample raster values*. We skip that here on purpose: with only a handful of flows it is clearer to read each elevation with **Identify Features** and type it into the sheet. The whole record is then four typed elevations and two typed distances per flow.

### The recording sheet

Everything goes into `qgis_lab_08/debris_flow_measurements.xlsx`. You only ever type the white columns:

| column | what you type | where it comes from |
|---|---|---|
| `flow_id` | integer | your inventory |
| `z_source` | elevation [m] | Identify on DTM at source |
| `z_beta` | elevation [m] | Identify on DTM at β-point |
| `z_toe` | elevation [m] | Identify on DTM at deposit toe |
| `L_beta` | distance [m] | Measure Line, source → β-point |
| `L_obs` | distance [m] | Measure Line, source → toe |
| `H` *(grey)* | — | auto: `z_source − z_toe` |
| `beta_deg` *(grey)* | — | auto: `atan((z_source − z_beta) / L_beta)` |

The Python cell in §8 reads this file directly, so once the sheet is filled there is nothing to copy and paste.

## 7. Theory — the α–β method 

The **α–β method** is the workhorse empirical run-out model in regional debris-flow and snow-avalanche hazard mapping. It rests on a simple observation: across many events in a given physiographic region, the **overall travel angle** α (from the source to the extreme distal point of the deposit) is **linearly related** to the **slope angle β** (from the source to the point where the channel gradient first flattens to a reference threshold, usually 20°). Steeper paths have steeper deposits, but the relationship is tight enough to predict α from β alone:

$$
\alpha \;=\; a\,\beta + b \qquad (1)
$$

Once α is known, the predicted run-out length *L* from source to toe is fixed by geometry:

$$
L \;=\; \frac{H}{\tan\alpha} \qquad (2)
$$

**Nora's updated coefficients.** The original Lied & Bakkehøi (1980) work fit $a \approx 0.96$, $b \approx -1.4^\circ$ on **13** Norwegian snow-avalanche events. Andreassen (2026, *unpublished MSc thesis*, NTNU) re-fit the relationship on **254 debris slides and debris flows** (*jordskred*) from the Norwegian Water Resources and Energy Directorate's (NVE) national landslide database — an almost twenty-fold expansion of the calibration dataset. Her result is:

$$
\boxed{\;\alpha \;=\; 0.99\,\beta \;-\; 3.53^\circ\;} \qquad (3)
$$

with $R^2 = 0.72$. The 95 % prediction interval for an individual event is approximately $\pm 6.4^\circ$ around the predicted α (essentially constant across the calibrated β range of 20–40°), which corresponds to a residual standard deviation of $\sigma_\alpha \approx 3.3^\circ$. This is wider than the prediction interval of the original 13-event fit — unsurprising given the much larger and more variable dataset — but the new model is statistically more robust because it is not driven by a handful of individual observations. Crucially, the 254 events are only the **calibration** set: Andreassen held out a further **109 independent *jordskred*** (363 in the full inventory) to **validate** the fitted relationship against data it had never seen. A model fitted on 254 events is only as trustworthy as its performance on that independent set — a check the original 13-event relationship never had.

Comparing the two fits, the Andreassen (2026) relationship generally predicts **slightly higher α** (i.e. **shorter run-outs**) than Lied & Bakkehøi (1980). The original equation is therefore the more conservative choice for hazard zoning, but the updated equation is the more representative choice for a *typical* Norwegian *jordskred*.

> ⚠️ Andreassen (2026) is **unpublished**. Use the coefficients in this notebook for the course exercise only. Do not circulate.

**Important caveats**

- The relationship is **regional**. Coefficients fitted on Norwegian *jordskred/debris flows* should not be exported to the Carpathians (or anywhere else) without re-fitting on a local dataset.
- It is **empirical**, not mechanical — no friction angle, no rheology, no entrainment.
- It gives a **single-number** run-out estimate. For probabilistic hazard zoning use $\alpha \pm 1\sigma$, $\pm 2\sigma$ to draw the orange / red / blue zones.
- It says nothing about lateral spread or impact pressure — only how far the flow travels.

In [ ]:
# Setup — only the alpha–beta calculation uses Python.
import numpy as np
import matplotlib.pyplot as plt

from style import apply_style, COLORS, save_figure
apply_style()

## 8. Predict run-out with Nora's α–β — small Python cell

The cell below reads `debris_flow_measurements.xlsx` directly, derives β and *H* from the columns you filled in QGIS, computes the predicted α and the predicted run-out length *L* from each source, and compares against the observed *L* you measured. Just make sure the spreadsheet is saved, then run the cell. Please note that we now use the library *Pandas*, which is needed to read Excel-files. If you haven't installed pandas, please run `pip install pandas`.

In [ ]:
# ---- Read the measurements you recorded in QGIS -----------------------
import pandas as pd
from pathlib import Path

XLSX = Path("qgis_lab_08/debris_flow_measurements.xlsx") # If this doesn't work, check the path and filename, and update as needed.
df = pd.read_excel(XLSX, sheet_name="measurements", header=3)

# keep only rows where the elevations and distances are actually filled in
needed = ["z_source", "z_beta", "z_toe", "L_beta", "L_obs"]
df = df.dropna(subset=needed).reset_index(drop=True)

flow_id  = df["flow_id"].to_numpy(dtype=int)
z_source = df["z_source"].to_numpy(dtype=float)
z_beta   = df["z_beta"].to_numpy(dtype=float)
z_toe    = df["z_toe"].to_numpy(dtype=float)
L_beta_m = df["L_beta"].to_numpy(dtype=float)
L_obs_m  = df["L_obs"].to_numpy(dtype=float)

# Derived directly from the recorded values (no raster extraction needed):
H_m      = z_source - z_toe                                   # fall height, m
beta_deg = np.degrees(np.arctan((z_source - z_beta) / L_beta_m))  # source->beta angle, deg

# ---- Nora's alpha-beta coefficients ----------------------------------
# Andreassen (2026, unpublished MSc thesis, NTNU): fit on n = 254 jordskred from the
# NVE database, validated on a further 109 independent events (363 total).
# Headline fit:  alpha = 0.989 * beta - 3.527 deg ,  R^2 = 0.72.
# 95% prediction interval ~ +/- 6.4 deg  ->  sigma_alpha ~ 3.27 deg.
# DO NOT CIRCULATE.
A_NORA  = 0.989           # slope of the alpha-beta line (dimensionless)
B_NORA  = -3.527          # intercept of the alpha-beta line (degrees)
S_ALPHA = 3.27            # residual standard deviation of alpha (degrees)

# ---- Predicted alpha and predicted runout length ---------------------
alpha_pred_deg = A_NORA * beta_deg + B_NORA
L_pred_m       = H_m / np.tan(np.radians(alpha_pred_deg))

# +/- 1 sigma envelope on alpha translates into a runout band on L
L_pred_lo_m = H_m / np.tan(np.radians(alpha_pred_deg + S_ALPHA))
L_pred_hi_m = H_m / np.tan(np.radians(alpha_pred_deg - S_ALPHA))

# ---- Pretty-print the comparison -------------------------------------
if len(flow_id) == 0:
    print("No completed rows yet — fill in debris_flow_measurements.xlsx and re-run.")
else:
    print(f"{'id':>3}  {'beta':>6}  {'alpha_p':>8}  {'L_obs':>8}  {'L_pred':>8}  {'L_lo':>8}  {'L_hi':>8}")
    for i in range(len(flow_id)):
        print(f"{flow_id[i]:>3d}  {beta_deg[i]:>6.1f}  {alpha_pred_deg[i]:>8.1f}  "
              f"{L_obs_m[i]:>8.0f}  {L_pred_m[i]:>8.0f}  {L_pred_lo_m[i]:>8.0f}  {L_pred_hi_m[i]:>8.0f}")

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

In [ ]:
# ---- Plot: observed vs. predicted runout length ---------------------
fig, ax = plt.subplots(figsize=(6.5, 6.0))

ax.errorbar(L_obs_m, L_pred_m,
            yerr=[L_pred_m - L_pred_lo_m, L_pred_hi_m - L_pred_m],
            fmt="o", color=COLORS["accent"], ecolor="grey",
            elinewidth=1.0, capsize=3, label=r"flows  ± 1σ on $\alpha$")

# 1:1 line
lim_max = np.nanmax([np.nanmax(L_obs_m), np.nanmax(L_pred_hi_m)]) * 1.1
if not np.isfinite(lim_max):   # no data pasted yet -> use a placeholder scale
    lim_max = 1.0
ax.plot([0, lim_max], [0, lim_max], color="black", lw=0.8, ls="--",
        label="1 : 1 (perfect prediction)")

ax.set_xlabel(r"observed runout length  $L_\mathrm{obs}$  [m]")
ax.set_ylabel(r"predicted runout length  $L_\mathrm{pred}$  [m]")
ax.set_title("α–β prediction vs. QGIS-mapped runout")
ax.set_xlim(0, lim_max); ax.set_ylim(0, lim_max)
ax.set_aspect("equal")
ax.legend(loc="lower right")

save_figure(fig, "alpha_beta_obs_vs_pred")
plt.show()

## 9.Questions for you to discuss

Things to comment on, in one paragraph, in your submission:

- Do the predicted run-out lengths fall within ± 1 σ of the observed ones, or are there systematic over-/under-predictions?
- If there is a systematic bias, what physiographic or geological feature of the study area might explain it? (Confined vs. unconfined channel, lithology, channel slope inflections, vegetation, entrainment of bed material…)
- Would you trust Nora's coefficients for hazard zoning in this catchment as-is, or would you want to re-fit them locally before drawing red/orange/yellow zones?

## Take-aways

- Operational landslide mapping happens in **QGIS**, not Python. Learning the desktop GIS workflow is part of the job.
- A multi-azimuth **hillshade** built from a 1 m bare-earth LiDAR DTM is the foundation of every modern landslide inventory.
- Digitising a small **inventory** — source point, track line, deposit polygon, consistent `flow_id` — turns visual interpretation into data you can analyse.
- The **α–β method** predicts run-out from two field-measurable angles: β (source to the 20° slope-break point) and, derived from it, α (source to deposit toe). Reclassifying a slope raster into 2° bins and displaying only the 19–21° class is a fast, reproducible way to locate the β-point on any flow path.
- Andreassen's (2026) re-fit on 254 Norwegian *jordskred* gives a more statistically robust α–β relationship than the original 13-event Lied & Bakkehøi (1980) fit, at the cost of a slightly wider prediction interval. The updated model also predicts *slightly shorter* run-outs on average — so it is less conservative for hazard zoning than the original.

## Voluntary exercises

1. Re-run the α–β prediction in §8 using the original Lied & Bakkehøi (1980) coefficients ($a = 0.96$, $b = -1.4^\circ$). How different are the predicted run-out lengths? In which direction? Which fit would you use for a hazard zonation, and why?
2. Pick one flow whose observed run-out falls well outside the ± 1 σ envelope. Open it in QGIS, look at the channel profile, and propose a physical explanation in 3–4 sentences.
3. Stack a second hillshade with a sun azimuth orthogonal to your first one. Identify one feature visible only in the second hillshade. What does this tell you about single-azimuth mapping?
4. Suppose you only have a 25 m EU-DEM tile instead of the 1 m LiDAR DTM. Which steps of this workflow still work, and which break down? Be specific.

## References

- Lied, K. & Bakkehøi, S. (1980). *Empirical calculations of snow-avalanche run-out distance based on topographic parameters.* Journal of Glaciology, 26(94), 165–177.
- Andreassen, N. (2026). *Fra 13 til 254: revidering og validering av α–β-modellen for utløpslengder av jordskred i Norge.* MSc thesis (TGB4945), Department of Geoscience and Petroleum, Norges Teknisk-Naturvitenskapelige Universitet (NTNU). Supervisor: O. Fredin; co-supervisor: A. Taurisano. **Unpublished — do not circulate.**
- Rickenmann, D. (2005). *Runout prediction methods.* In: Jakob, M. & Hungr, O. (eds), *Debris-flow Hazards and Related Phenomena*, Springer–Praxis, 305–324.
- QGIS Development Team. *QGIS Geographic Information System.* Open Source Geospatial Foundation Project. [qgis.org](https://qgis.org).